# ViTrueVoice — Đánh Giá Baseline Checkpoint B Trên Kênh Zalo Dual-Profile (Inference-Only)

Notebook này thực thi quy trình chấm điểm độc lập (inference-only) trên toàn bộ **11.450 mẫu Dev Zalo** bằng Checkpoint B (`best_model_B.pt`).

### 5 Cổng Kiểm Soát Chất Lượng Bắt Buộc:
1. **Cổng 1 — Khóa Artifacts & Hash**: Mount 2 kernel source; SHA-256 Checkpoint B (`a921c339…`) và `dev_zalo.csv` (`012b2721…`) khớp 100%.
2. **Cổng 2 — Giải nén Chọn Lọc (Bijective Selective Unzip)**: Giải nén đủ 6.786 file gốc từ `vitruevoice_data_processed.zip`; 0 file thiếu, 0 file ghi đè.
3. **Cổng 3 — Khóa Mô hình (strict=True)**: Nạp 295/295 keys Checkpoint B; đặt `model.eval()`, `torch.inference_mode()`; tuyệt đối không train.
4. **Cổng 4 — Suy luận Toàn Bộ 11.450 Mẫu (Zero Silent Skip)**: Chấm đủ 11.450 mẫu 5 codec (AAC: 2.332, Opus: 2.332, Clean: 2.332, AMR: 2.227, G.711: 2.227); fail-fast nếu thiếu bất kỳ mẫu nào.
5. **Cổng 5 — Báo cáo & Kiểm định Hai Lớp**: Tính EER, minDCF, FRR@0, FAR@0 và Cluster Bootstrap 95% CI theo `base_utt`; bắt buộc mỗi codec có đủ cả bona fide lẫn spoof mới công nhận baseline.

In [ ]:
# 1. Kiểm tra môi trường GPU & Nạp trước cấu hình HuggingFace
import os
import sys
import time
import shutil
import hashlib
import zipfile
import subprocess
import json
import zlib
import base64
from pathlib import Path

import torch
import torchaudio
import soundfile as sf
import numpy as np
import pandas as pd
from transformers import AutoConfig, Wav2Vec2Model
from huggingface_hub import hf_hub_download

print('=' * 60)
print('⚙️ MÔI TRƯỜNG THỰC THI KAGGLE GPU')
print('=' * 60)
print(f'• Python: {sys.version}')
print(f'• PyTorch: {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'• GPU: {torch.cuda.get_device_name(0)}')
    print(f'• VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
else:
    raise SystemError('❌ Kernel yêu cầu GPU nhưng không tìm thấy CUDA device!')

EXPECTED_HF_REVISION = '69e9000591623e5a4fc2f502407860bcdc0de0b2'
print(f'🌐 Pre-warm cấu hình Wav2Vec2 tại commit bất biến: {EXPECTED_HF_REVISION}...')
t0 = time.time()
hf_cfg = AutoConfig.from_pretrained('nguyenvulebinh/wav2vec2-base-vietnamese-250h', revision=EXPECTED_HF_REVISION)
print(f'✅ Đã tải và khóa cấu hình HuggingFace thành công ({time.time() - t0:.2f}s)')


In [ ]:
# 2. Bung mã nguồn src/ và dev_zalo.csv từ payload bất biến
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')

b64_src = '...'
src_files = json.loads(zlib.decompress(base64.b64decode(b64_src.encode('ascii'))).decode('utf-8'))
for fpath_str, content in src_files.items():
    dest = working_dir / fpath_str
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(content, encoding='utf-8')
print(f'✅ Đã bung {len(src_files)} tệp mã nguồn src/ vào {working_dir}')

b64_manifest = '...'
manifest_raw = zlib.decompress(base64.b64decode(b64_manifest.encode('ascii')))
manifest_dest = working_dir / 'dev_zalo.csv'
manifest_dest.write_bytes(manifest_raw)

EXPECTED_MANIFEST_SHA = '012b272137795cb2740fd3fb9afeed49859e91dccb4c14d115921cbbad7585c1'
actual_manifest_sha = hashlib.sha256(manifest_dest.read_bytes()).hexdigest()
print(f'📄 Manifest: {manifest_dest}')
print(f'   - SHA-256 thực tế:  {actual_manifest_sha}')
print(f'   - SHA-256 kỳ vọng:  {EXPECTED_MANIFEST_SHA}')
assert actual_manifest_sha == EXPECTED_MANIFEST_SHA, '❌ Sai lệch SHA-256 dev_zalo.csv!'

df_dev = pd.read_csv(manifest_dest)
print(f'📊 Phân bổ Dev Zalo: {len(df_dev)} dòng')
assert len(df_dev) == 11450, f'❌ Số dòng không phải 11450: {len(df_dev)}'
assert sum(df_dev['label'] == 'bonafide') == 10490, '❌ Sai số lượng bonafide (kỳ vọng 10490)'
assert sum(df_dev['label'] == 'spoof') == 960, '❌ Sai số lượng spoof (kỳ vọng 960)'
assert not df_dev['speaker_id'].str.startswith('SPK_').any(), '❌ Phát hiện SPK_xxxx trong Dev Zalo!'
assert 'VieNeu-TTS' not in df_dev['tts_generator'].values, '❌ Phát hiện VieNeu-TTS trong Dev Zalo!'

if str(working_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(working_dir.resolve()))
from src.models.ssl_aasist import SSLAASIST
from src.preprocessing.audio_loader import load_audio
from src.training.dataset import fit_length
from src.evaluation.metrics import compute_eer, compute_min_dcf
print('✅ Nạp thành công các module từ src/')


In [ ]:
# 3. CỔNG 1 — Xác minh Nguồn Kernel & Checkpoint B SHA-256
def compute_sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

EXPECTED_CKPT_SHA = 'a921c3397946335c9543528993eafde80ac0d42decf431101635806b5c56bd9f'

# Tìm Checkpoint B
ckpt_candidates = (
    list(Path('/kaggle/input').rglob('*best_model_B*.pt')) +
    list(Path('/kaggle/input').rglob('*run_B*best_model*.pt')) +
    list(Path('/kaggle/input').rglob('*best_model*.pt')) +
    [
        Path('/kaggle/input/vietrue-ssl-training/best_model_B.pt'),
        Path('/kaggle/input/vietrue-ssl-training/run_B/best_model.pt'),
        Path('out/kaggle_ssl_out/best_model_B.pt'),
    ]
)
ckpt_path = None
for c in ckpt_candidates:
    if c.exists() and c.is_file() and c.stat().st_size > 10000000:
        if compute_sha256(c) == EXPECTED_CKPT_SHA:
            ckpt_path = c
            break
assert ckpt_path is not None, f'❌ Không tìm thấy Checkpoint B khớp SHA-256!'
print(f'📦 Checkpoint B: {ckpt_path}')
print(f'   - SHA-256: {compute_sha256(ckpt_path)}')

# Tìm file ZIP dữ liệu processed
zip_candidates = (
    list(Path('/kaggle/input').rglob('*vitruevoice*.zip')) +
    list(Path('/kaggle/input').rglob('*.zip')) +
    [
        Path('/kaggle/input/vietrue-model-training/vitruevoice_data_processed.zip'),
    ]
)
zip_path = None
for z in zip_candidates:
    if z.exists() and z.is_file() and z.stat().st_size > 1000000:
        zip_path = z
        break
assert zip_path is not None, f'❌ Không tìm thấy file ZIP dữ liệu âm thanh!'
print(f'📦 Data ZIP: {zip_path} ({zip_path.stat().st_size / (1024**2):.1f} MB)')
print('🏁 CỔNG 1: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 4. CỔNG 2 — Giải nén chọn lọc đúng các tệp audio gốc cần thiết
t_unzip_start = time.perf_counter()

# Thu thập toàn bộ đường dẫn audio gốc trong ZIP cần thiết:
# Clean, G.711, AMR: dùng trực tiếp đường dẫn audio_path
# AAC, Opus: dùng file master clean của cùng base_utt
clean_rows = df_dev[df_dev['codec'] == 'clean_16khz']
base_to_clean_path = dict(zip(clean_rows['base_utt'], clean_rows['audio_path']))

required_rel_paths = set()
for _, row in df_dev.iterrows():
    c = row['codec']
    if c in ['clean_16khz', 'g711_alaw', 'amr_nb']:
        required_rel_paths.add(row['audio_path'].replace('\\', '/').lstrip('/'))
    else:
        # Proxy AAC/Opus lấy từ clean master
        master_p = base_to_clean_path[row['base_utt']].replace('\\', '/').lstrip('/')
        required_rel_paths.add(master_p)

print(f'🎯 Tổng số tệp audio gốc cần trích xuất từ ZIP: {len(required_rel_paths)}')

with zipfile.ZipFile(zip_path, 'r') as zf:
    namelist_set = set(zf.namelist())
    extract_targets = []
    missing_in_zip = []
    
    for rel_p in required_rel_paths:
        candidates_zip = [rel_p, f'data/{rel_p}', rel_p.replace('data/', '')]
        found_entry = None
        for cand in candidates_zip:
            if cand in namelist_set:
                found_entry = cand
                break
        if found_entry is None:
            missing_in_zip.append(rel_p)
        else:
            dest_target = (working_dir / rel_p).resolve()
            extract_targets.append((found_entry, dest_target))

    print(f'   - Đã ánh xạ trong ZIP: {len(extract_targets)} / {len(required_rel_paths)} tệp')
    if missing_in_zip:
        print(f'❌ Thiếu {len(missing_in_zip)} tệp trong ZIP! Mẫu đầu: {missing_in_zip[:5]}')
        raise FileNotFoundError(f'Thiếu {len(missing_in_zip)} tệp trong ZIP!')
        
    print('🚀 Đang thực hiện Selective Unzip...')
    for zip_entry, dest_target in extract_targets:
        dest_target.parent.mkdir(parents=True, exist_ok=True)
        if not dest_target.exists():
            with zf.open(zip_entry) as src, open(dest_target, 'wb') as dst:
                shutil.copyfileobj(src, dst)

print(f'✅ Giải nén chọn lọc hoàn tất trong {time.perf_counter() - t_unzip_start:.1f}s!')

# Kiểm tra 100% tệp gốc tồn tại
missing_files = [p for p in required_rel_paths if not (working_dir / p).is_file()]
print(f'🔍 Kiểm tra đĩa: {len(required_rel_paths) - len(missing_files)} / {len(required_rel_paths)} tồn tại')
assert len(missing_files) == 0, f'❌ Vẫn còn {len(missing_files)} tệp không tìm thấy sau khi giải nén!'
print('🏁 CỔNG 2: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 5. CỔNG 3 — Nạp Checkpoint B strict=True & Khóa eval mode
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🤖 Đang nạp Checkpoint B: {ckpt_path} lên {device}...')
ckpt_dict = torch.load(ckpt_path, map_location=device, weights_only=False)

model_kwargs = dict(ckpt_dict.get('model_kwargs', {}))
model_kwargs['ssl_config'] = hf_cfg
model = SSLAASIST(**model_kwargs)

state_dict = ckpt_dict['model'] if 'model' in ckpt_dict else ckpt_dict['model_state_dict']
load_result = model.load_state_dict(state_dict, strict=True)
print(f'✅ Khớp 295/295 keys thành công: {load_result}')

model.to(device)
model.eval()
for param in model.parameters():
    param.requires_grad = False

assert not model.training, '❌ Mô hình chưa ở trạng thái eval()!'
print('🏁 CỔNG 3: HOÀN TOÀN ĐẠT CHUẨN (strict=True, eval-only)!')


In [ ]:
# 6. CỔNG 4 — Suy luận toàn bộ 11.450 mẫu Zalo (Fail-Fast, Zero Silent Skip)
def transcode_aac_adts_stream(raw_wav_bytes: bytes) -> bytes:
    cmd_enc = ['ffmpeg', '-nostdin', '-v', 'error', '-i', 'pipe:0', '-ar', '48000', '-ac', '1', '-c:a', 'aac', '-b:a', '76k', '-f', 'adts', 'pipe:1']
    proc_enc = subprocess.run(cmd_enc, input=raw_wav_bytes, capture_output=True, check=True)
    cmd_dec = ['ffmpeg', '-nostdin', '-v', 'error', '-i', 'pipe:0', '-f', 'wav', '-ar', '16000', '-ac', '1', 'pipe:1']
    proc_dec = subprocess.run(cmd_dec, input=proc_enc.stdout, capture_output=True, check=True)
    return proc_dec.stdout

def transcode_opus_ogg_stream(raw_wav_bytes: bytes) -> bytes:
    cmd_enc = ['ffmpeg', '-nostdin', '-v', 'error', '-i', 'pipe:0', '-c:a', 'libopus', '-b:a', '24k', '-f', 'ogg', 'pipe:1']
    proc_enc = subprocess.run(cmd_enc, input=raw_wav_bytes, capture_output=True, check=True)
    cmd_dec = ['ffmpeg', '-nostdin', '-v', 'error', '-i', 'pipe:0', '-f', 'wav', '-ar', '16000', '-ac', '1', 'pipe:1']
    proc_dec = subprocess.run(cmd_dec, input=proc_enc.stdout, capture_output=True, check=True)
    return proc_dec.stdout

# Pre-load clean audio bytes in memory for fast proxy transcoding
print('⚡ Đang nạp sẵn master clean bytes vào RAM để tối ưu tốc độ transcode...')
clean_bytes_map = {}
for base_utt, rel_p in base_to_clean_path.items():
    abs_p = working_dir / rel_p.replace('\\', '/').lstrip('/')
    clean_bytes_map[base_utt] = abs_p.read_bytes()
print(f'✅ Đã nạp {len(clean_bytes_map)} clean masters vào RAM.')

rng = np.random.default_rng(0)
scored_records = []
t_infer_start = time.perf_counter()

print(f'🚀 Bắt đầu suy luận tuần tự trên đúng {len(df_dev)} mẫu Dev Zalo...')
with torch.inference_mode():
    for idx, row in df_dev.iterrows():
        codec = row['codec']
        base_utt = row['base_utt']
        label_int = 1 if str(row['label']).lower() in ['bonafide', '1'] else 0
        
        wav_16k = None
        if codec in ['clean_16khz', 'g711_alaw', 'amr_nb']:
            p = working_dir / row['audio_path'].replace('\\', '/').lstrip('/')
            if not p.is_file():
                raise FileNotFoundError(f'❌ Dòng {idx}: Thiếu tệp audio {p}!')
            wav_16k = load_audio(p, target_sr=16000)
        elif codec == 'aac_zalo_76k':
            c_bytes = clean_bytes_map[base_utt]
            proxy_bytes = transcode_aac_adts_stream(c_bytes)
            wav_16k = load_audio(proxy_bytes, target_sr=16000)
        elif codec == 'opus_zalo_24k':
            c_bytes = clean_bytes_map[base_utt]
            proxy_bytes = transcode_opus_ogg_stream(c_bytes)
            wav_16k = load_audio(proxy_bytes, target_sr=16000)
            
        if wav_16k is None:
            raise RuntimeError(f'❌ Dòng {idx} ({row["utt_id"]}): Giải mã âm thanh thất bại!')
            
        t = fit_length(wav_16k, 64000, train=False, rng=rng)
        tensor = torch.from_numpy(t).float().unsqueeze(0).to(device)
        score = model.score(tensor).item()
        if np.isnan(score) or np.isinf(score):
            raise ValueError(f'❌ Dòng {idx}: Điểm số suy luận bị NaN/Inf!')
            
        rec = dict(row)
        rec['score'] = score
        rec['label_int'] = label_int
        scored_records.append(rec)
        
        if (idx + 1) % 1000 == 0 or (idx + 1) == len(df_dev):
            elapsed = time.perf_counter() - t_infer_start
            rate = (idx + 1) / elapsed
            print(f'   - Đã chấm {idx + 1}/{len(df_dev)} mẫu ({rate:.1f} mẫu/s, trôi qua {elapsed:.1f}s)...')

scored_df = pd.DataFrame(scored_records)
assert len(scored_df) == 11450, f'❌ Số lượng mẫu đã chấm không đủ 11450: {len(scored_df)}'
out_scores_path = working_dir / 'dev_zalo_scores.csv'
scored_df.to_csv(out_scores_path, index=False)
print(f'✅ Đã lưu bảng điểm đầy đủ 11.450 mẫu vào {out_scores_path}')
print('🏁 CỔNG 4: HOÀN TOÀN ĐẠT CHUẨN!')


In [ ]:
# 7. CỔNG 5 — Báo cáo & Kiểm định Metrics Hai Lớp
def cluster_bootstrap_ci(subset: pd.DataFrame, n_boot: int = 1000, seed: int = 42) -> dict:
    clusters, cluster_inv = np.unique(subset['base_utt'].values, return_inverse=True)
    n_c = len(clusters)
    if n_c < 5 or subset['label_int'].nunique() < 2:
        return {}
    cluster_indices = [np.where(cluster_inv == i)[0] for i in range(n_c)]
    y_true_all = subset['label_int'].to_numpy(dtype=int)
    y_score_all = subset['score'].to_numpy(dtype=float)
    rng = np.random.default_rng(seed)
    boot_eers, boot_frrs, boot_fars = [], [], []
    for _ in range(n_boot):
        sampled_c = rng.choice(n_c, size=n_c, replace=True)
        idx = np.concatenate([cluster_indices[c] for c in sampled_c])
        y_true = y_true_all[idx]
        y_score = y_score_all[idx]
        if len(np.unique(y_true)) < 2:
            continue
        try:
            e, _ = compute_eer(y_score, y_true)
            boot_eers.append(e * 100.0)
        except Exception:
            pass
        n_b = np.sum(y_true == 1)
        n_s = np.sum(y_true == 0)
        if n_b > 0:
            boot_frrs.append(float(np.sum((y_true == 1) & (y_score < 0.0)) / n_b * 100.0))
        if n_s > 0:
            boot_fars.append(float(np.sum((y_true == 0) & (y_score >= 0.0)) / n_s * 100.0))
    def ci(arr):
        if len(arr) == 0:
            return (float('nan'), float('nan'))
        return (float(np.percentile(arr, 2.5)), float(np.percentile(arr, 97.5)))
    return {'eer_ci': ci(boot_eers), 'frr_ci': ci(boot_frrs), 'far_ci': ci(boot_fars)}

metric_rows = []
for c_name in ['aac_zalo_76k', 'opus_zalo_24k', 'clean_16khz', 'g711_alaw', 'amr_nb']:
    c_df = scored_df[scored_df['codec'] == c_name]
    y_true = c_df['label_int'].to_numpy().astype(int)
    y_score = c_df['score'].to_numpy().astype(float)
    n_bona = int(np.sum(y_true == 1))
    n_spoof = int(np.sum(y_true == 0))
    has_both = (n_bona > 0 and n_spoof > 0)
    assert has_both, f'❌ Codec {c_name} thiếu một trong hai lớp (bona={n_bona}, spoof={n_spoof})!'
    
    eer, eer_thr = compute_eer(y_score, y_true)
    min_dcf = compute_min_dcf(y_score, y_true)
    frr_0 = float(np.sum((y_true == 1) & (y_score < 0.0)) / n_bona * 100.0)
    far_0 = float(np.sum((y_true == 0) & (y_score >= 0.0)) / n_spoof * 100.0)
    
    ci_res = cluster_bootstrap_ci(c_df, n_boot=1000)
    eer_ci = ci_res.get('eer_ci', (float('nan'), float('nan')))
    frr_ci = ci_res.get('frr_ci', (float('nan'), float('nan')))
    far_ci = ci_res.get('far_ci', (float('nan'), float('nan')))
    
    metric_rows.append({
        'codec': c_name,
        'n_total': len(c_df),
        'n_bonafide': n_bona,
        'n_spoof': n_spoof,
        'is_valid': True,
        'eer_pct': eer * 100.0,
        'eer_ci_95': f'[{eer_ci[0]:.2f}, {eer_ci[1]:.2f}]',
        'eer_threshold': eer_thr,
        'min_dcf': min_dcf,
        'frr_theta_0': frr_0,
        'frr_ci_95': f'[{frr_ci[0]:.2f}, {frr_ci[1]:.2f}]',
        'far_theta_0': far_0,
        'far_ci_95': f'[{far_ci[0]:.2f}, {far_ci[1]:.2f}]',
    })

metrics_df = pd.DataFrame(metric_rows)
out_metrics_path = working_dir / 'zalo_baseline_metrics.csv'
metrics_df.to_csv(out_metrics_path, index=False)
print('=' * 80)
print('📊 BÁO CÁO BASELINE CHECKPOINT B TRÊN TẬP DEV ZALO ĐẦY ĐỦ (11.450 MẪU)')
print('=' * 80)
print(metrics_df[['codec', 'n_bonafide', 'n_spoof', 'eer_pct', 'eer_ci_95', 'frr_theta_0', 'far_theta_0', 'min_dcf']].to_string(index=False))

# Paired Deltas so với clean
clean_map = scored_df[scored_df['codec'] == 'clean_16khz'].set_index('base_utt')['score'].to_dict()
delta_rows = []
for p_codec in ['aac_zalo_76k', 'opus_zalo_24k', 'g711_alaw', 'amr_nb']:
    sub = scored_df[scored_df['codec'] == p_codec].copy()
    sub['clean_score'] = sub['base_utt'].map(clean_map)
    sub = sub.dropna(subset=['clean_score'])
    diff = sub['score'] - sub['clean_score']
    delta_rows.append({
        'codec': p_codec,
        'n_paired': len(diff),
        'mean_delta': float(diff.mean()),
        'std_delta': float(diff.std()),
        'median_delta': float(diff.median()),
        'min_delta': float(diff.min()),
        'max_delta': float(diff.max()),
    })
delta_df = pd.DataFrame(delta_rows)
out_delta_path = working_dir / 'zalo_paired_deltas.csv'
delta_df.to_csv(out_delta_path, index=False)
print()
print('📊 ĐỘ LỆCH ĐIỂM SỐ GHÉP CẶP (PAIRED DELTAS vs CLEAN):')
print(delta_df.to_string(index=False))
print()
print('🏁 CỔNG 5: BASELINE ĐƯỢC CÔNG NHẬN HỢP LỆ VỚI ĐỦ CẢ HAI LỚP TRÊN 5 CODEC!')

# Dọn dẹp tệp âm thanh tạm thời trên disk để output Kaggle gọn nhẹ
data_dir = working_dir / 'data'
if data_dir.exists():
    shutil.rmtree(data_dir, ignore_errors=True)
